In [1]:
# %matplotlib inline
# %matplotlib notebook
# %matplotlib widget
import glob
import os
import rasterio
import numpy as np
from matplotlib import pyplot as plt

In [2]:
imgDir = '/mnt/warehouse/sahel/planet_medres_normalized_analytic_2019-06_2019-11_mosaic'
cropPredDir = '/mnt/warehouse/sahel/pred_2019'

subfolders = [f.path for f in os.scandir(cropPredDir) if f.is_dir()]
subfolders.sort()
print (len(subfolders),subfolders[-1:])

506 ['/mnt/warehouse/sahel/planet_medres_normalized_analytic_2019-06_2019-11_mosaic/00016_00040']


In [3]:


cutoffFolders = []
for i in subfolders[:-1]:
    cutoffFolders.append([f.path for f in os.scandir(i) if f.is_dir() and os.path.basename(f)=='cutouts'])
print (len(cutoffFolders),cutoffFolders[0])

123 []


In [ ]:
for i in range(0,len(subfolders)):
    predFolder = f'{subfolders[i]}/pred'
    outDir = f'{subfolders[i]}/withPred'
    if not os.path.exists(outDir):
        os.makedirs(outDir)
    splitFiles = glob.glob(f'{cutoffFolders[i][0]}/split*.tif',recursive = True)
    for j in range(0,len(splitFiles)):
        searchId = os.path.basename(splitFiles[j])
        predFile = glob.glob(f'{predFolder}/crop_{searchId}',recursive = True)[0]
        print (predFile,splitFiles[j])
    

        pred = rasterio.open(predFile)
        predArr = pred.read(1)
        mosaicFile = splitFiles[j]
        img = rasterio.open(mosaicFile)
        imgArr = img.read([1,2,3,4,5])
        print (predArr.shape, imgArr.shape)

        # Write the NDVI image
        meta = img.meta

        meta.update(driver='GTiff')
        meta.update(dtype=rasterio.int16)
        meta.update(count=6)

        with rasterio.open(f'{outDir}/pred_{os.path.basename(mosaicFile)}', 'w', **meta) as dst:
            dst.write(np.append(imgArr.astype(np.int16), [predArr.astype(np.int16)], axis=0))